# Agentic RAG

## 导入一个你可用的模型模型

In [1]:
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat",
                api_key = 'YOUR_DEEPSEEK_API_KEY',
                base_url = 'https://api.deepseek.com')

## 建立向量数据库连接

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# embedding_model_path = '/data/LLM/maple77/xiaobu-embedding-v2'
embedding_model_path = r"C:\Users\yanchen\.cache\huggingface\hub\models--BAAI--bge-small-zh-v1.5\snapshots\7999e1d3359715c523056ef9478215996d62a620"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_path,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="./rag_cache/chroma_db",  # 与创建时的目录完全一致（相对/绝对路径都可）
    embedding_function=embeddings     # 必须传入相同的 Embedding 实例
)

## 改写 query 和提取关键词 Agent

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field

class expand_and_keyword_foramt(BaseModel):
    """对用户的query进行改写和提取关键词的结果"""
    expand_query: str = Field(..., description="对query进行扩写/改写的结果，可以不改写")
    keyword: str = Field(..., description="在query中提取一个最重要的简短关键词")

@tool
def expand_and_keyword(query):
    """
    对用户输入的query进行改写和提取一个最重要的关键词
    :param query: str,  必填 - 是用户的输入，在此基础上进行扩写和提取关键词。
    :return:expand_query: str,  必填 - 对用户query进行扩写/改写的结果，可以不改写。
    :return:keyword: str, 对用户的输入提取一个最重要的关键词。
    """
    
    expand_and_keyword_conversation = [
        {"role": "system", "content": "你是一个乐于助人的助手，负责给用户的query进行改写/扩写，同时在query中提取一个最重要的关键词"},
        {"role": "user", "content": query},
    ]
    
    model_with_structure = model.with_structured_output(expand_and_keyword_foramt)
    response = model_with_structure.invoke(expand_and_keyword_conversation)
    expand_query = response.expand_query
    keyword = response.keyword

    return expand_query,keyword

## 检索过程 Agent

In [4]:
@tool
def retrieval_augment(query, keyword):
    """
    【RAG检索增强】基于用户输入的查询语句和核心关键词，从知识库中检索相关文档并增强结果。
    核心逻辑：先按query检索Top10文档→若有keyword则调整匹配得分（含关键词的文档得分-0.1）→按得分升序排序→拼接前5条文档内容返回。

    :param query: str, 必填 - 用户输入的自然语言查询语句，作为基础检索条件
    :param keyword: str, 必填 - 从用户输入中提取的核心关键词，用于优化检索结果权重
    :return:related_doc: str, 按优化后得分排序库文档内容的检索结果，以双换行符分隔拼接后的字符串
    """

    # print(query, keyword)
    # 进行检索
    docs = vectorstore.similarity_search_with_score(query, k=10)
    docs = [list(item) for item in docs] # 元组改为列表，对欧拉距离进行改写
    
    if keyword:
        for doc in docs:
            # 如果关键词在文本中，就让得分更少（更相似）
            if keyword in doc[0].page_content:
                doc[-1] -= 0.1
                
        # 对关键词搜索后的检索片段进行排序        
        docs = sorted(docs, key=lambda x: x[1])
        
    # 只取前五个片段
    docs = docs[:5]
    related_doc = '\n\n'.join([text[0].page_content for text in docs])
    
    return related_doc

## 对 chunks 进行精读

In [5]:
class summary_related_doc_foramt(BaseModel):
    """对用户query相关的片段进行筛选"""
    summary_related_doc_res: str = Field(..., description="对用户query相关的片段进行筛选")


@tool
def summary_related_doc(query, related_doc):
    """
    片段精读。根据用户query，在related_doc原始片段中把与query相关的片段进行摘取，不要进行改写，仅把相关部分进行筛选复制。

    :param query: str, 必填 - 用户输入的自然语言查询语句，作为基础检索条件
    :param related_doc: str, 必填 - 原始的文档片段
    :return:summary_related_doc_res: str, 筛选后的片段
    """
    
    query_and_related_doc = '这是用户的query：{}\n\n这是原文的众多片段：{}'.format(query, related_doc)
    summary_related_doc_conversation = [
            {"role": "system", "content": "你是一个乐于助人的助手，现在有很多原文片段，你需要对用户query相关的片段进行筛选，不要改动原文，只把相关的片段进行复制筛选即可"},
            {"role": "user", "content": query_and_related_doc},
        ]
    
    model_with_structure = model.with_structured_output(summary_related_doc_foramt)
    response = model_with_structure.invoke(summary_related_doc_conversation)
    summary_related_doc_res = response.summary_related_doc_res
    return summary_related_doc_res

## 让模型进行自主选择是否调用工具及定义重复次数

In [6]:
system_prompt = '''
你是一名助人为乐的助手，你会根据用户的问题进行判断：

- 如果用户问题与校规不相关，那么就直接进行回答，不要调用工具。

- 如果用户问题与校规相关，就需要进行RAG检索，然后根据检索到的相关文档进行回答，具体步骤如下：
    1. 可选择：如果用户描述不清楚，你需要调用expand_and_keyword函数对用户问题进行改写并提取关键词
    2. 必须进行：调用retrieval_augment函数，基于用户输入的查询语句和核心关键词，从知识库中检索相关文档
    3. 可选择：若retrieval_augment检索后的文章大于1000字，可函数调用summary_related_doc函数对retrieval_augment检索的文档进行精读，抽取筛选与query相关的字段
    4. 根据精读的内容对用户的原始query进行回答；若没有找到答案，请再次对问题进行改写和关键词提取，然后再次进行RAG检索（最多重复两次）
    5. 若重复两次还没找到答案，请回答：没有在文档中检索到相关内容，所以无法准确回答
'''

## 创建 RAG Agent

In [7]:
class agent_result(BaseModel):
    """对用户query进行回答"""
    agent_answer: str = Field(..., description="对用户query进行回答")

tools = [expand_and_keyword, retrieval_augment, summary_related_doc]

agent = create_agent(model=model,
                     tools=tools,
                     system_prompt = system_prompt,
                     response_format= agent_result,
                    )

## 询问简单问题，不会走 RAG 流程

In [8]:
query = '你是谁？'

response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)
response

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='2f5aa731-bf54-479b-99f8-53048c00ca4b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 984, 'total_tokens': 1060, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 984}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '0d487ad7-a818-457f-bf80-aed2d429ea0e', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1860-0727-79d0-bdcf-db4a0104536e-0', tool_calls=[{'name': 'agent_result', 'args': {'agent_answer': '你好！我是你的智能助手，很高兴为你服务！我可以帮你解答各种问题，包括校规相关的查询（需要检索知识库），以及其他各类日常问题。有什么需要帮助的吗？😊'}, 'id': 'call_00_MQxFAWm8DjAzS9rgRKsc3879', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to

## 询问知识库相关问题

In [9]:
query = '两周未参加学校活动会被退学吗'

response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

for res in response['messages']:
    print(res.name,end = '\n\n')

None

None

expand_and_keyword

None

retrieval_augment

None

summary_related_doc

None

agent_result



In [ ]:
response

笔记：
可以发现搜索tool得到的结果比较长，传入总结tool的结果比较短

也就是意味着，tool调用是由模型决定的，同时对应的参数也不是完全管道传入，可能有压缩

## 查询不在知识库中的问题

In [11]:
query = '如果想要换班级怎么走流程？'

response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

In [12]:
response

{'messages': [HumanMessage(content='如果想要换班级怎么走流程？', additional_kwargs={}, response_metadata={}, id='b63eca2b-63cf-4e67-8729-4533180c9552'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 989, 'total_tokens': 1032, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 896}, 'prompt_cache_hit_tokens': 896, 'prompt_cache_miss_tokens': 93}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'fbc634d7-df0e-4956-bcaf-5bc09fd3b089', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1863-97a9-7ae2-a154-31db19109b5f-0', tool_calls=[{'name': 'expand_and_keyword', 'args': {'query': '想要换班级怎么走流程'}, 'id': 'call_00_CUPfU8BeMnNryYaEPfoG6384', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 989, 'output_tokens': 43, 'total_tokens': 103

In [13]:
for res in response['messages']:
    print(res.name,end = '\n\n')

None

None

expand_and_keyword

None

retrieval_augment

None

summary_related_doc

None

expand_and_keyword

None

retrieval_augment

None

summary_related_doc

None

agent_result



## 使用中间件限制工具调用次数

In [15]:
system_prompt = '''
你是一名助人为乐的助手，你会根据用户的问题进行判断：

- 如果用户问题与校规不相关，那么就直接进行回答，不要调用工具。

- 如果用户问题与校规相关，就需要进行RAG检索，然后根据检索到的相关文档进行回答，具体步骤如下：
    1. 可选择：如果用户描述不清楚，你需要调用expand_and_keyword函数对用户问题进行改写并提取关键词
    2. 必须进行：调用retrieval_augment函数，基于用户输入的查询语句和核心关键词，从知识库中检索相关文档
    3. 可选择：若retrieval_augment检索后的文章大于1000字，可函数调用summary_related_doc函数对retrieval_augment检索的文档进行精读，抽取筛选与query相关的字段
    4. 根据精读的内容对用户的原始query进行回答；若没有找到答案，请再次对问题进行改写和关键词提取，然后再次进行RAG检索（最多重复两次）
    5. 若还没找到答案，请回答：没有在文档中检索到相关内容，所以无法准确回答
'''

In [16]:
# 下面为wrap_tool_middleware的官方实现的使用方法

from langchain.agents.middleware import ToolCallLimitMiddleware

class agent_result(BaseModel):
    """对用户query进行回答"""
    agent_answer: str = Field(..., description="对用户query进行回答")

tools = [expand_and_keyword, retrieval_augment, summary_related_doc]

agent = create_agent(model=model,
                     tools=tools,
                     system_prompt = system_prompt,
                     response_format= agent_result,
                     middleware=[
                            # Global limit
                            ToolCallLimitMiddleware(run_limit=10),
                            # Tool-specific limit
                            ToolCallLimitMiddleware(tool_name="expand_and_keyword",run_limit=2),
                            ToolCallLimitMiddleware(tool_name="retrieval_augment",run_limit=2),
                            ToolCallLimitMiddleware(tool_name="summary_related_doc",run_limit=2)
                     ]
                    )

In [17]:
query = '如果想要换班级怎么走流程？'

response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

for res in response['messages']:
    print(res.name,end = '\n\n')

None

None

expand_and_keyword

None

retrieval_augment

None

summary_related_doc

None

agent_result

